<a href="https://colab.research.google.com/github/doomguy0991/cx_transformers/blob/main/Lecture_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


In [ ]:
import os

# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Setting up Colab environment...")

    # 1. Clone the repository
    repo_url = "https://github.com/doomguy0991/cx_transformers.git"
    # Only clone if the directory doesn't exist yet
    if not os.path.exists("/content/cx_transformers"):
        !git clone $repo_url /content/cx_transformers

    # 2. Change working directory to the notebook's location so relative paths for libraries work imports
    %cd "/content/cx_transformers"

    print("Setup complete. You can now run the rest of the notebook.")
else:
    print("Running locally or out of Colab. No setup needed.")


# Lecture 2: The Intuition Behind Self-Attention

## Table of Contents
- [Section 1: The Core NLP Problem: Text Vectorization](#section-1-the-core-nlp-problem-text-vectorization)
  - [1.1 The Requirement for NLP Applications](#11-the-requirement-for-nlp-applications)
  - [1.2 The Evolution of Vectorization](#12-the-evolution-of-vectorization)
- [Section 2: The Power of Word Embeddings](#section-2-the-power-of-word-embeddings)
  - [2.1 What is Semantic Meaning?](#21-what-is-semantic-meaning)
  - [2.2 The Geometry of Meaning](#22-the-geometry-of-meaning)
  - [2.3 What do the Dimensions Represent?](#23-what-do-the-dimensions-represent)
- [Section 3: The Fatal Flaw of Static Embeddings](#section-3-the-fatal-flaw-of-static-embeddings)
  - [3.1 The "Apple" Dilemma](#31-the-apple-dilemma)
  - [3.2 The Problem with "Static"](#32-the-problem-with-static)
  - [3.3 The Goal: Contextual Embeddings](#33-the-goal-contextual-embeddings)
- [Section 4: Enter Self-Attention: Contextual Embeddings](#section-4-enter-self-attention-contextual-embeddings)
  - [4.1 The Self-Attention Function](#41-the-self-attention-function)
  - [4.2 The Magic Result](#42-the-magic-result)
  - [Conclusion and Next Steps](#conclusion-and-next-steps)

---

## Section 1: The Core NLP Problem: Text Vectorization

> *"To understand why we need Self-Attention, we must first understand the fundamental problem of NLP: How do we convert words into numbers?"*

Before we dive into the mathematical mechanics of Self-Attention (Queries, Keys, Values), we need to establish a clear understanding of exactly **what** Self-Attention is trying to achieve. If you understand the goal of Self-Attention, understanding the Transformers architecture becomes incredibly easy.

### 1.1 The Requirement for NLP Applications
Take any Natural Language Processing (NLP) application:
*   Sentiment Analysis
*   Named Entity Recognition (NER)
*   Machine Translation

What is the single most important requirement for these tasks? The input is always a sequence of words (a sentence, a review, a document). However, **computers do not understand words; they only understand numbers.** 

Therefore, the first and most critical step in NLP is figuring out how to efficiently represent words as numbers. This process is formally known as **Vectorization**.

### 1.2 The Evolution of Vectorization

Early research in NLP heavily focused on this exact problem. Over time, several basic techniques emerged.

#### 1. One-Hot Encoding
The simplest and most original technique. Here is how it works:
1.  Identify the entire vocabulary (all unique words) in your dataset.
2.  Represent each word as a vector of zeros, with a single `1` at the index corresponding to that word.

**Example:** 
Suppose we have two sentences: 
1. `"mat cat"`
2. `"mat cat rat rat"`

Our unique vocabulary is: `["mat", "cat", "rat"]` (Size = 3).
*   `mat` $\rightarrow [1, 0, 0]$
*   `cat` $\rightarrow [0, 1, 0]$
*   `rat` $\rightarrow [0, 0, 1]$

Sentence 1 (`"mat cat"`) becomes: `[[1, 0, 0], [0, 1, 0]]`

**The Flaw:** This method is highly inefficient. If your vocabulary has 10,000 words, every single word is represented by a 10,000-dimensional vector containing mostly zeros (sparse representation). 

#### 2. Bag of Words (BoW)
To slightly improve upon One-Hot Encoding, Bag of Words doesn't just check if a word exists, it counts **how many times** the word appears in the sentence.

Using the same vocabulary `["mat", "cat", "rat"]`:
*   Sentence 1 (`"mat cat"`): `[1, 1, 0]` (mat: 1, cat: 1, rat: 0)
*   Sentence 2 (`"mat cat rat rat"`): `[1, 1, 2]` (mat: 1, cat: 1, rat: 2)

**The Flaw:** While slightly more informative, BoW entirely destroys the **order** of the words. `"mat cat"` and `"cat mat"` would result in the exact same vector. Furthermore, words like "the" or "is" might dominate the counts without adding meaning (which is why techniques like TF-IDF were later introduced).


In [ ]:
# Python Demonstration: One-Hot Encoding vs Bag of Words
import numpy as np

sentences = [
    "mat cat",
    "mat cat rat rat"
]

# 1. Build Vocabulary
vocab = []
for sentence in sentences:
    for word in sentence.split():
        if word not in vocab:
            vocab.append(word)

print(f"Vocabulary: {vocab}\n")

# 2. One-Hot Encoding for the word "mat"
one_hot_mat = np.zeros(len(vocab), dtype=int)
one_hot_mat[vocab.index("mat")] = 1
print(f"One-Hot Vector for 'mat': {one_hot_mat}\n")

# 3. Bag of Words for Sentence 2
bow_sentence_2 = np.zeros(len(vocab), dtype=int)
for word in sentences[1].split():
    bow_sentence_2[vocab.index(word)] += 1

print(f"Bag of Words for 'mat cat rat rat': {bow_sentence_2}")


## Section 2: The Power of Word Embeddings

> *"The real fun began when a technique called Word Embeddings arrived. They were advanced because they had the power to capture semantic meaning."*

To fix the extreme inefficiency of One-Hot Encoding and the context-blindness of Bag of Words, the NLP community created **Word Embeddings**. 

### 2.1 What is Semantic Meaning?
When we say "Semantic Meaning", we mean that a word is converted into numbers in such a way that the *meaning* of the word, and the context it is generally used in, is physically captured by the math.

To create these embeddings:
1. We take a massive dataset (like all of Wikipedia).
2. We pass it through a neural network.
3. The neural network learns the relationships between words by seeing which words frequently appear next to each other.
4. It outputs an **$n$-dimensional vector** for every single word (usually $n=256$, $512$, or $1024$).

### 2.2 The Geometry of Meaning
Let's simplify and pretend our neural network outputs a **5-dimensional vector** for each word. 

After training, the network might represent the word `King` as:
$$\text{King} = [0.6, 0.2, 0.1, 0.0, 0.9]$$

And the word `Queen` as:
$$\text{Queen} = [0.6, 0.3, 0.1, 0.1, 0.8]$$

**The core geometric intuition is this:** Because these two words share very similar semantic meanings, their mathematical vectors will look almost identical. If you plot them in a 5-dimensional space, the points for "King" and "Queen" will be clustered very close together.

Conversely, if we look at the vector for `Cricketer`, it might look entirely different:
$$\text{Cricketer} = [0.1, 0.9, 0.9, 0.0, 0.1]$$
In our geometric space, the `Cricketer` point will be very far away from the `King` and `Queen` points.

### 2.3 What do the Dimensions Represent?
While neural networks are "black boxes" and don't label their dimensions for us, we can conceptually understand them as representing different **abstract features** of the language.

For example, in our 5-dimensional space:
*   **Dimension 1 (Royalty):** `King` and `Queen` will have very high values here, while `Cricketer` will have a near-zero value.
*   **Dimension 2 (Athleticism):** `Cricketer` will have a huge spike here, while `King` and `Queen` will have low values.
*   **Dimension 3 (Humanity):** `King`, `Queen`, and `Cricketer` are all humans! So, all three words will have a high value in this specific dimension.

By judging the similarity across hundreds of these hidden conceptual dimensions, Word Embeddings allow computers to mathematically "understand" language.


In [ ]:
# Python Demonstration: Visualizing Word Embeddings
import numpy as np
import matplotlib.pyplot as plt

# Let's create mock embeddings based on our lecture intuition
# Dimensions: [Royalty, Athleticism, Humanity]
vocab = ["King", "Queen", "Cricketer", "Apple"]

embeddings = {
    "King":      np.array([0.9, 0.1, 0.9]),
    "Queen":     np.array([0.9, 0.2, 0.9]),
    "Cricketer": np.array([0.1, 0.9, 0.9]),
    "Apple":     np.array([0.0, 0.0, 0.0]) # Not royal, not an athlete, not human
}

# We will visualize them using a heatmap
embedding_matrix = np.array(list(embeddings.values()))

fig, ax = plt.subplots(figsize=(6, 4))
cax = ax.imshow(embedding_matrix, cmap='YlGnBu', aspect='auto')

# Formatting the plot
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(["Royalty", "Athleticism", "Humanity"])
ax.set_yticks(np.arange(len(vocab)))
ax.set_yticklabels(vocab)
ax.set_title("Word Embeddings Heatmap")

plt.colorbar(cax, label="Feature Strength")
plt.tight_layout()
plt.show()

# Notice how visually similar King and Queen are across the rows, 
# while Cricketer and Apple are completely different!


## Section 3: The Fatal Flaw of Static Embeddings

> *"Word embeddings are made once and used again and again. This means they are static. And sometimes, this capability can cause harm."*

Word embeddings are incredibly powerful, but they suffer from one fundamental, critical flaw: **They capture the *average* meaning of a word across the entire training dataset, not the specific contextual meaning.**

### 3.1 The "Apple" Dilemma
Imagine we train a new word embedding model. To keep things simple, let's say our neural network only outputs a 2-dimensional vector for each word:
1.  **Dimension 1:** Taste
2.  **Dimension 2:** Technology

Now, let's feed the model some sentences containing the word `Apple`:
*   *"An apple a day keeps the doctor away."* $\rightarrow$ Model learns `Apple` is related to **Taste**.
*   *"Apple is healthy."* $\rightarrow$ Model becomes more confident about **Taste**.
*   *"Apple makes great phones."* $\rightarrow$ Model suddenly sees `Apple` related to **Technology**.

If our training dataset of 10,000 sentences contains 9,000 sentences where `Apple` is a fruit, and 1,000 sentences where `Apple` is a tech company, the final learned vector for the word `Apple` might look like this:
$$\text{Apple} = [0.9, 0.3]$$
*(High on Taste, low on Technology).*

### 3.2 The Problem with "Static"
The core problem is that once this embedding is trained, it becomes **static**. It is locked in. Wherever you use the word `Apple` in the future, the computer will inject the exact same mathematical vector: $[0.9, 0.3]$.

Now imagine you build an English-to-Hindi translation application. You feed it the following sentence:
> *"Apple launched a new phone while I was eating an orange."*

What happens?
1. The machine looks up the embedding for `Apple`: $[0.9, 0.3]$ (Fruit).
2. The machine sees the word `orange` and gets further confused, thinking the whole sentence is about fruit.
3. The translation completely fails because the machine has fundamentally misunderstood the context of the word `Apple` in this specific sentence.

> [!WARNING]
> Because static embeddings are locked into an **average** representation, they completely break down when a word has multiple meanings depending on its surrounding sentence (Polysemy). 

### 3.3 The Goal: Contextual Embeddings
To solve this, we don't want a static vector. We want a dynamic, **Contextual Embedding**. 

Ideally, as soon as the algorithm reads the words `"launched"` and `"phone"` in the same sentence as `"Apple"`, it should automatically shift the vector mathematically:
*   Decrease the "Taste" dimension.
*   Massively increase the "Technology" dimension.

It should dynamically realize: *"In this specific sentence, Apple is a tech company, not a fruit."*


## Section 4: Enter Self-Attention: Contextual Embeddings

> *"Self-Attention is a mechanism that takes static embeddings as input, and generates smart, contextual embeddings as output."*

We finally arrive at the exact reason why the Transformer architecture was invented. To solve the fatal flaw of static word embeddings, we need a mathematical function that can look at a whole sentence and adjust the embedding of each word based on its neighbors.

This function is called **Self-Attention**.

### 4.1 The Self-Attention Function
You can think of Self-Attention as a "black box" function for now:
$$ \text{Contextual Embeddings} = \text{SelfAttention}(\text{Static Embeddings}) $$

Here is exactly what happens when you pass the problematic sentence into the Self-Attention mechanism:
1.  **Input:** You feed the entire sentence into the function simultaneously. This means you send the *static* embedding for `"Apple"`, the *static* embedding for `"launched"`, the *static* embedding for `"phone"`, and the *static* embedding for `"orange"`.
2.  **Internal Calculations:** The mechanism allows every word to mathematically "look" at every other word in the sentence. `"Apple"` looks at `"launched"` and `"phone"`.
3.  **Output:** For every single input word, the function spits out exactly one new output embedding. 

But these output embeddings are **Smart Contextual Embeddings**.

### 4.2 The Magic Result
Because the mechanism saw the words `"launched"` and `"phone"` surrounding `"Apple"`, it automatically performs a mathematical shift on Apple's embedding:

*   **Input (Static):** $\text{Apple} = [0.9, 0.3]$ *(Fruit)*
*   **Output (Contextual):** $\text{Apple} = [0.2, 0.9]$ *(Tech Company)*

It is smart enough to completely ignore the word `"orange"` later in the sentence because the immediate grammatical context strongly implies technology.

### Conclusion and Next Steps
By understanding this progression—from One-Hot Encoding, to Static Word Embeddings, to Contextual Embeddings—you now fully understand the **"What"** and the **"Why"** of Self-Attention.

In the next lecture, we will open up the "black box" and look at the **"How"**. We will learn the exact mathematical formulas inside the Self-Attention mechanism that allow it to shift these vectors, including the famous concept of **Query, Key, and Value vectors**.
